In [2]:
#@title Import Library & Gabungkan Data
import os
import re
import unicodedata
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

os.makedirs("./data/processed", exist_ok=True)
os.makedirs("./models", exist_ok=True)

df_hf = pd.read_csv("hf_dataset.csv")
df_yt = pd.read_csv("youtube_comments_dataset.csv")

df = pd.concat([df_hf, df_yt], ignore_index=True)
df["id"] = range(1, len(df) + 1)
df["label"] = df["label"].astype(int)

print("Total data awal setelah digabung:", len(df))

Total data awal setelah digabung: 11247


In [3]:
#@title Data Cleaning

def bersihkan_teks(teks):
    # Ubah font Unicode menjadi huruf biasa
    teks = unicodedata.normalize("NFKC", str(teks))

    teks = teks.lower()
    # Hapus tautan dan mention
    teks = re.sub(r"http\S+|www\.\S+|@\w+", " ", teks)
    # Ubah kata/angka penyamaran pada kata campuran huruf-angka (misal g4cor -> gacor)
    kamus_angka = str.maketrans({"4": "a", "3": "e", "1": "i", "0": "o", "5": "s", "7": "t"})
    kata_list = []
    for kata in teks.split():
        if any(c.isalpha() for c in kata) and any(c.isdigit() for c in kata):
            kata = kata.translate(kamus_angka)
        kata_list.append(kata)
    teks = " ".join(kata_list)
    # Sisakan huruf alfabet (a-z) dan spasi
    teks = re.sub(r"[^a-z\s]", " ", teks)
    teks = re.sub(r"\s+", " ", teks).strip()
    return teks

df["clean_text"] = df["text_caption"].apply(bersihkan_teks)

# Hapus teks yang menjadi kosong setelah dibersihkan dan hapus duplikat
df_clean = df[df["clean_text"] != ""].drop_duplicates(subset=["clean_text"]).reset_index(drop=True)

print("Total data bersih setelah hapus teks kosong & duplikat:", len(df_clean))
print("\nProporsi kelas setelah cleaning:")
print(df_clean["label"].value_counts())

Total data bersih setelah hapus teks kosong & duplikat: 10558

Proporsi kelas setelah cleaning:
label
0    6098
1    4460
Name: count, dtype: int64


In [4]:
#@title data splitting

# Gabungkan platform dan label sebagai kunci stratifikasi agar proporsinya merata
df_clean["strat_key"] = df_clean["platform"] + "_" + df_clean["label"].astype(str)

# 80% Train dan 20% Sisa (Validation + Test)
train_df, temp_df = train_test_split(
    df_clean, test_size=0.20, random_state=42, stratify=df_clean["strat_key"]
)

# Bagi 20% Sisa menjadi 10% Validation dan 10% Test
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df["strat_key"]
)

print(f"Jumlah Train set : {len(train_df)} sampel")
print(f"Jumlah Val set   : {len(val_df)} sampel")
print(f"Jumlah Test set  : {len(test_df)} sampel")

kolom_simpan = ["id", "clean_text", "platform", "label"]
train_df[kolom_simpan].to_csv("./data/processed/train.csv", index=False)
val_df[kolom_simpan].to_csv("./data/processed/val.csv", index=False)
test_df[kolom_simpan].to_csv("./data/processed/test.csv", index=False)

Jumlah Train set : 8446 sampel
Jumlah Val set   : 1056 sampel
Jumlah Test set  : 1056 sampel


In [5]:
#@title Feature Extraction & Transformasi

# TF-IDF untuk Naive Bayes & Random Forest
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(train_df["clean_text"])
X_val_tfidf = tfidf.transform(val_df["clean_text"])
X_test_tfidf = tfidf.transform(test_df["clean_text"])

# 2. Tokenisasi & Padding untuk LSTM
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df["clean_text"])

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(train_df["clean_text"]), maxlen=100, padding="post")
X_val_seq = pad_sequences(tokenizer.texts_to_sequences(val_df["clean_text"]), maxlen=100, padding="post")
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(test_df["clean_text"]), maxlen=100, padding="post")

print("Dimensi fitur TF-IDF Train :", X_train_tfidf.shape)
print("Dimensi sekuens LSTM Train :", X_train_seq.shape)

# Simpan vectorizer & tokenizer
joblib.dump(tfidf, "./models/tfidf_vectorizer.joblib")
joblib.dump(tokenizer, "./models/lstm_tokenizer.joblib")
print("Data preparation selesai dan seluruh file berhasil disimpan!")

Dimensi fitur TF-IDF Train : (8446, 5000)
Dimensi sekuens LSTM Train : (8446, 100)
Data preparation selesai dan seluruh file berhasil disimpan!
